# Exploration 03 — standardization, normalization and effective dimension

> **Exploratory** (CLAUDE.md, "Exploration"). **Inputs:** the headline-legal feature sets from notebook 02 (Z and R
> only). **Data:** the training pool P for description; cross-validation inside the training sets S_(s,N) for
> every choice. No test set is read.

**Part 3 of the classical analysis.** Input: `data/processed/explore_features.parquet` and the feature catalog
(notebook 02), plus the EDA's findings on distribution shape (notebook 01). Output:
`results/explore03_preprocessing.json`, the feature scaling and target transform that notebooks 04 and 05 use.

**Choice rules, fixed before looking at any CV number:**
- **scaling:** the candidate with the lowest mean CV MAE over every (scale-sensitive model, legal feature set,
  seed, N) cell. All candidates share the same cells and folds, so the comparison is paired;
- **target transform:** the same rule, over ridge, RBF kernel ridge and XGBoost on the two richest legal sets.

Runtime: about 15 minutes. Every cell is safe to re-run.

In [ ]:
# Environment check: stops here, with setup instructions, unless this kernel is the project
# environment (README.md, "Setup"). Version differences from requirements.txt only warn.
try:
    from qm9dipole.provenance import check_environment
except ModuleNotFoundError as e:
    raise RuntimeError(
        "The qm9dipole package is not installed in this kernel's environment. Follow "
        "README.md (Setup), then select that environment as the notebook kernel."
    ) from e

print(check_environment())

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.pipeline import Pipeline

from qm9dipole import REPO_ROOT
from qm9dipole.analysis import summarize_curves
from qm9dipole.evaluate import learning_curve
from qm9dipole.explore import PREPROCESSING_PATH, feature_sets, load_catalog, load_config, load_features
from qm9dipole.models import classical
from qm9dipole.plots import (AXIS, INK_SECONDARY, MODEL_COLORS, MODEL_LABELS, MUTED, SEQUENTIAL, SERIES,
                             SURFACE, use_style)
from qm9dipole.preprocess import scaling_steps
from qm9dipole.provenance import config_hash, git_hash, save_result
from qm9dipole.splits import load_splits

use_style()
FIG_DIR = REPO_ROOT / "figures"
CFG = load_config()
features = load_features()
catalog = load_catalog()
SETS = feature_sets(catalog)
LEGAL = ["composition", "cm", "structure", "cm+structure"]
splits = load_splits()
y = features["mu"]
pool = features[features["pool_analysis"]]  # one twin per duplicate pair, for pool-wide description
print(f"{len(features):,} pool molecules ({len(pool):,} for pool-wide description); legal sets: "
      + ", ".join(f"{s} ({len(SETS[s])})" for s in LEGAL))

## 1. Why scaling matters here

Ridge's penalty, the RBF kernel's distances, PCA and the quantum feature map's rotation angles all treat a unit
step in every feature as equally important. Raw features differ in scale by orders of magnitude, so without scaling
the largest-variance features would dominate all four. Tree models (random forest, XGBoost) split on one feature at
a time and are unaffected by any monotone rescaling.

In [ ]:
X_all = pool[SETS["cm+structure"]]
spread = X_all.std()
print(f"standard deviation across the 60 legal features: {spread.min():.3g} to {spread.max():.3g} "
      f"(ratio {spread.max() / spread[spread > 0].min():,.0f})")
display(spread.sort_values(ascending=False).iloc[[0, 1, 2, -3, -2, -1]].rename("std").to_frame()
        .style.format("{:.3g}").set_caption("largest and smallest spreads"))

## 2. What each scaling does to the shape

Five candidates (`src/qm9dipole/preprocess.py`): z-score (**standard**), median/IQR (**robust**), log of the
skewed columns then z-score (**log_standard**), a fitted power transform (**yeo_johnson**), and a rank map to a
normal (**quantile_normal**). Fit on the pool for description only (models refit inside every fold). Two
consequences matter:
- **skewness:** long tails let a few molecules dominate distances and the ridge fit;
- **extremes:** the quantum feature map turns each value into a rotation angle γ·x. Angles wrap around every 2π, so
  a value far out in the tail aliases onto ordinary ones. The share beyond ±3 measures that risk.

In [ ]:
CANDIDATES = CFG["standardization"]["candidates"]
SIGMA = CFG["quantum"]["angle_sigma"]
legal_X = pool[SETS["cm+structure"]].to_numpy()
shape_rows, scaled_pool = [], {}
for sc in CANDIDATES:
    Z = Pipeline(scaling_steps(sc, CFG["standardization"]["skew_threshold"])).fit_transform(legal_X)
    scaled_pool[sc] = Z
    sk = pd.DataFrame(Z).skew().abs()
    shape_rows.append({"scaling": sc, "median |skew|": sk.median(), "max |skew|": sk.max(),
                       "max |value|": np.abs(Z).max(), f"share beyond ±{SIGMA}": (np.abs(Z) > SIGMA).mean()})
shape = pd.DataFrame(shape_rows).set_index("scaling")
display(shape.style.format({"median |skew|": "{:.2f}", "max |skew|": "{:.1f}", "max |value|": "{:.1f}",
                            f"share beyond ±{SIGMA}": "{:.2%}"}))

show = ["inertia_moment_3", "cm_01", "bond_dipole_norm", "n_N"]
idx = [SETS["cm+structure"].index(c) for c in show]
fig, axes = plt.subplots(len(show), len(CANDIDATES), figsize=(15, 8.4))
for i, (col, j) in enumerate(zip(show, idx)):
    for k, sc in enumerate(CANDIDATES):
        ax = axes[i, k]
        ax.hist(scaled_pool[sc][:, j], bins=40, color=SERIES[0], edgecolor=SURFACE, linewidth=0.3)
        ax.tick_params(labelsize=7)
        ax.grid(axis="x", visible=False)
        if i == 0:
            ax.set_title(sc, fontsize=10)
        if k == 0:
            ax.set_ylabel(col, fontsize=9)
fig.suptitle("Four legal features under each scaling (rows: features; columns: scalings)", x=0.01, ha="left",
             fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "explore03_scaling_shapes.png")
plt.show()

## 3. Choose the scaling by cross-validation

Ridge and RBF kernel ridge (the scale-sensitive models) on every legal set, plus the 8-component compressed variants
the quantum model will use, at every (seed, N), with the same folds as notebook 04. All other settings are as in
notebook 04 (target standardized, hyperparameters tuned by CV).

In [ ]:
frames = {name: features[SETS[name]] for name in LEGAL}
frames |= {f"{name}/pca8": features[SETS[name]] for name in ("cm", "cm+structure")}
compare = tuple(CFG["standardization"]["compare_models"])

start = time.perf_counter()
scaling_runs = []
for sc in CANDIDATES:
    def make_specs(seed, fname, sc=sc):
        return classical.specs(seed, compressed=fname.endswith("/pca8"), models=compare, scaling=sc)
    res, _ = learning_curve(make_specs, frames, y, splits.train, progress=False)
    scaling_runs.append(res.assign(scaling=sc))
    print(f"{sc:16s} done ({(time.perf_counter() - start) / 60:.1f} min)")
scaling_cv = pd.concat(scaling_runs, ignore_index=True)

mean_mae = scaling_cv.groupby("scaling")["mae_D"].mean().loc[CANDIDATES]
best_scaling = mean_mae.idxmin()
paired = scaling_cv.pivot_table(index=["seed", "n_train", "model", "features"], columns="scaling", values="mae_D")
wins = paired.eq(paired.min(axis=1), axis=0).mean().loc[CANDIDATES]
vs_standard = paired.sub(paired["standard"], axis=0).mean().loc[CANDIDATES]
choice = pd.DataFrame({"mean CV MAE (D)": mean_mae, "Δ vs standard (D)": vs_standard, "share of cells won": wins})
display(choice.style.format({"mean CV MAE (D)": "{:.4f}", "Δ vs standard (D)": "{:+.4f}", "share of cells won": "{:.0%}"})
        .set_caption(f"{len(paired)} paired cells; chosen: {best_scaling}"))
by_n = scaling_cv.pivot_table(index=["n_train", "model"], columns="scaling", values="mae_D", aggfunc="mean")[CANDIDATES]
display(by_n.style.format("{:.3f}").highlight_min(axis=1, props="font-weight: bold").set_caption("mean CV MAE (D) by N and model"))
print(f"chosen scaling: {best_scaling}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
pv = scaling_cv[scaling_cv["features"] == "cm+structure"].pivot_table(index=["n_train", "model"], columns="scaling",
                                                                   values="mae_D", aggfunc="mean")[CANDIDATES]
x = np.arange(len(pv))
for k, sc in enumerate(CANDIDATES):
    # Dots, not bars: the differences are ~0.02 D on ~0.9 D, and bars would need a truncated axis.
    ax.scatter(x + (k - 2) * 0.12, pv[sc], color=SERIES[k], s=46, zorder=3, edgecolors=SURFACE, linewidths=1, label=sc)
ax.set_xticks(x, [f"N={n}" + chr(10) + MODEL_LABELS[m] for n, m in pv.index], fontsize=8)
ax.set(ylabel="CV MAE (D), mean over seeds", title="Scalings on cm+structure (60 legal features)")
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.22))
ax.grid(axis="x", visible=False)
fig.savefig(FIG_DIR / "explore03_scaling_cv.png")
plt.show()

## 4. Choose the target transform by cross-validation

The EDA found |μ| right-skewed (1.73) and √|μ| nearly symmetric (−0.26). A model fit on a transformed target is
inverted back to debye before scoring, so only accuracy in debye counts. Every candidate also standardizes, so
kernel ridge's lack of an intercept never biases it toward 0 D. Three model families (linear, kernel, trees) on the
two richest legal sets, using the scaling chosen above.

In [ ]:
TARGETS = CFG["standardization"]["target_candidates"]
target_frames = {name: features[SETS[name]] for name in ("structure", "cm+structure")}
target_models = ("ridge", "rbf_krr", "xgb")
start = time.perf_counter()
target_runs = []
for tg in TARGETS:
    def make_specs(seed, fname, tg=tg):
        return classical.specs(seed, models=target_models, scaling=best_scaling, target=tg)
    res, _ = learning_curve(make_specs, target_frames, y, splits.train, progress=False)
    target_runs.append(res.assign(target=tg))
    print(f"{tg:9s} done ({(time.perf_counter() - start) / 60:.1f} min)")
target_cv = pd.concat(target_runs, ignore_index=True)

t_mae = target_cv.groupby("target")["mae_D"].mean().loc[TARGETS]
t_rmse = target_cv.groupby("target")["rmse_D"].mean().loc[TARGETS]
best_target = t_mae.idxmin()
display(pd.DataFrame({"mean CV MAE (D)": t_mae, "mean CV RMSE (D)": t_rmse}).style.format("{:.4f}")
        .set_caption(f"chosen by MAE: {best_target}"))
display(target_cv.pivot_table(index=["n_train", "model"], columns="target", values="mae_D", aggfunc="mean")[TARGETS]
        .style.format("{:.3f}").highlight_min(axis=1, props="font-weight: bold").set_caption("mean CV MAE (D) by N and model"))
print(f"chosen target transform: {best_target}")

## 5. Principal components with the chosen scaling

The 8-qubit feature map takes 8 numbers, so the compressed variant keeps the first 8 principal components. Three
questions: how much variance do 8 components keep; do the high-variance directions carry the |μ| signal; and what
does compression cost a model? Variance and relevance are described on the pool. The cost is measured by CV inside
S_(0,1000), with PCA refit in every fold.

In [ ]:
from sklearn.decomposition import PCA

K = CFG["quantum"]["n_qubits"]
pca_sets = {"cm": "Coulomb spectrum (29)", "structure": "structure (31)", "cm+structure": "cm+structure (60)"}
pcas, scores = {}, {}
for name in pca_sets:
    Z = Pipeline(scaling_steps(best_scaling, CFG["standardization"]["skew_threshold"])).fit_transform(pool[SETS[name]])
    pcas[name] = PCA().fit(Z)
    scores[name] = pcas[name].transform(Z)
kept = pd.DataFrame({f"variance kept by {K}": [p.explained_variance_ratio_[:K].sum() for p in pcas.values()]},
                    index=list(pca_sets.values()))
display(kept.style.format("{:.1%}"))

n_show = 20
yp = pool["mu"].to_numpy()
rel = {name: [abs(pd.Series(scores[name][:, k]).corr(pd.Series(yp), method="spearman")) for k in range(n_show)]
       for name in pca_sets}
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 3.9))
for color, (name, label) in zip(SERIES, pca_sets.items()):
    cum = np.cumsum(pcas[name].explained_variance_ratio_)
    ax0.plot(np.arange(1, len(cum) + 1), cum, color=color, marker="o", markersize=3, label=label)
ax0.axvline(K, color=INK_SECONDARY, linewidth=1)
ax0.annotate(f"{K} components = {K} qubits", (K, 0.05), xytext=(5, 0), textcoords="offset points",
             color=INK_SECONDARY, fontsize=9)
ax0.set(xlabel="principal components kept", ylabel="share of variance kept", ylim=(0, 1.02),
        title=f"Variance kept ({best_scaling} scaling)")
ax0.legend(loc="center right")
ax1.bar(np.arange(1, n_show + 1), rel["cm+structure"], width=0.72,
        color=[SERIES[0] if k < K else MUTED for k in range(n_show)])
ax1.axvline(K + 0.5, color=INK_SECONDARY, linewidth=1)
ax1.set(xlabel="principal component (cm+structure)", ylabel="|Spearman ρ| with |μ|", xticks=range(1, n_show + 1, 2),
        title=f"How strongly each component tracks |μ| (first {K} in blue)")
ax1.grid(axis="x", visible=False)
fig.savefig(FIG_DIR / "explore03_pca.png")
plt.show()

pc_std = pd.DataFrame({label: scores[name][:, :K].std(axis=0) for name, label in pca_sets.items()},
                      index=[f"PC{k + 1}" for k in range(K)])
display(pc_std.T.style.format("{:.2f}").set_caption(
    "standard deviation of each of the first 8 component scores: the spread a rotation angle γ·x would inherit"))

In [ ]:
from qm9dipole.analysis import pca_cv_curve
from qm9dipole.evaluate import kfold

SEED, N = 0, 1000
ids = splits.train[SEED][N]
ys = y.loc[ids].to_numpy()
folds = kfold(len(ids), SEED)
ks = [1, 2, 4, 6, 8, 10, 12, 16, 20, 24, 30]
curves = []
for name in ("cm", "cm+structure"):
    X = features.loc[ids, SETS[name]].to_numpy()
    for model in ("ridge", "rbf_krr"):
        c = pca_cv_curve(X, ys, [k for k in ks if k < X.shape[1]], folds, model=model, seed=SEED,
                         scaling=best_scaling, target=best_target)
        curves.append(c.assign(features=name, model=model))
curves = pd.concat(curves, ignore_index=True)
save_result(curves.assign(params=curves["params"].map(str)), "explore03_compression_cost", seed=SEED, n_train=N,
            scaling=best_scaling, target=best_target, label="exploratory; legal inputs; CV inside S_(0,1000)")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.8), sharey=True)
for ax, name in zip(axes, ("cm", "cm+structure")):
    for model in ("ridge", "rbf_krr"):
        c = curves[(curves["features"] == name) & (curves["model"] == model)]
        pc, full = c[c["pca"]], c[~c["pca"]].iloc[0]
        ax.plot(pc["k"], pc["mae_D"], color=MODEL_COLORS[model], marker="o", label=f"{MODEL_LABELS[model]}, k components")
        ax.scatter([full["k"]], [full["mae_D"]], color=MODEL_COLORS[model], marker="D", s=40, zorder=3,
                   edgecolors=SURFACE, linewidths=1, label=f"{MODEL_LABELS[model]}, all {full['k']} (no PCA)")
    ax.axvline(K, color=INK_SECONDARY, linewidth=1)
    ax.set(xlabel="principal components kept (k)", title=pca_sets[name])
axes[0].set_ylabel("CV MAE (D), S_(0,1000)")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=4)
fig.savefig(FIG_DIR / "explore03_compression_cost.png")
plt.show()
at_k = curves[(curves["k"] == K) | ~curves["pca"]].pivot_table(index=["features", "model"], columns="pca", values="mae_D")
at_k.columns = ["all features", f"{K} components"]
display(at_k.assign(**{"cost of compressing (D)": at_k.iloc[:, 1] - at_k.iloc[:, 0]}).style.format("{:.3f}"))

## 6. Effective dimension: how many independent directions do the features span?

A set of 60 features can behave like far fewer if they are correlated. Four measures on the scaled pool:
- **participation ratio** PR = (Σλ)² / Σλ² of the covariance eigenvalues: the number of equally strong directions
  with the same overall spread;
- **components for 90 / 95 / 99 %** of the variance;
- **condition number** √(λ_max / λ_min): infinite when some feature is an exact linear combination of others;
- **variance inflation factor** (VIF) per feature: 1 / (1 − R²) from regressing it on the rest. Above ~10, a linear
  model cannot separate its effect.

This is the dimension of the inputs. The effective number of parameters a fitted model uses is measured in
notebook 04.

In [ ]:
from qm9dipole.complexity import components_for_share, condition_number, participation_ratio, vif

dim_rows = []
for name in LEGAL:
    Z = Pipeline(scaling_steps(best_scaling, CFG["standardization"]["skew_threshold"])).fit_transform(pool[SETS[name]])
    lam = np.linalg.eigvalsh(np.cov(Z, rowvar=False))[::-1]
    dim_rows.append({"feature set": name, "features": Z.shape[1], "participation ratio": participation_ratio(lam),
                     "k for 90%": components_for_share(lam, 0.90), "k for 95%": components_for_share(lam, 0.95),
                     "k for 99%": components_for_share(lam, 0.99), "condition number": condition_number(lam)})
dims = pd.DataFrame(dim_rows).set_index("feature set")
display(dims.style.format({"participation ratio": "{:.1f}", "condition number": "{:.3g}"}))

Zs = Pipeline(scaling_steps(best_scaling, CFG["standardization"]["skew_threshold"])).fit_transform(pool[SETS["structure"]])
v = vif(Zs, SETS["structure"]).sort_values(ascending=False)
print(f"structure set: {np.isinf(v).sum()} features with infinite VIF (exact linear combinations), "
      f"{((v > 10) & np.isfinite(v)).sum()} more above 10, {(v <= 10).sum()} at or below 10")
display(v.head(12).to_frame().style.format("{:.3g}"))

## 7. Save the choices for notebooks 04 and 05

In [ ]:
choice_out = {
    "scaling": best_scaling, "target": best_target,
    "rule": "lowest mean CV MAE over all paired (model, feature set, seed, N) cells",
    "scaling_cv_mean_mae": mean_mae.round(5).to_dict(), "target_cv_mean_mae": t_mae.round(5).to_dict(),
    "n_qubits": K, "variance_kept_by_n_qubits": kept.iloc[:, 0].round(4).to_dict(),
    "effective_dimension": dims[["features", "participation ratio", "k for 95%"]].round(2).reset_index().to_dict("records"),
    "config_hash": config_hash(CFG), "git_hash": git_hash(),
}
PREPROCESSING_PATH.write_text(json.dumps(choice_out, indent=1) + "\n")
save_result(scaling_cv, "explore03_scaling_cv", label="exploratory; legal inputs; CV inside S_(s,N)")
save_result(target_cv, "explore03_target_cv", scaling=best_scaling, label="exploratory; legal inputs; CV inside S_(s,N)")
print(f"wrote {PREPROCESSING_PATH.relative_to(REPO_ROOT).as_posix()}: scaling = {best_scaling}, target = {best_target}")

## Summary

1. **Scaling is mandatory for most of the pipeline.** Feature spreads differ by a factor of 22,643 (moments of
   inertia vs the last Coulomb eigenvalues), and ridge, kernels, PCA and quantum angles would all be dominated by
   the large ones. Trees are indifferent.
2. **Shapes.** Yeo-Johnson makes the features most symmetric (median |skewness| 1.13 → 0.15) and leaves the fewest
   values beyond ±3 (0.74%, vs 1.27% for z-scores). The quantile map bounds everything at ±5.2 but sends the zeros
   of every count feature to −5.2, so 39% of all values land beyond ±3. That is the opposite of what an angle
   encoding needs.
3. **Scaling chosen by the rule fixed in advance: Yeo-Johnson.** It has the lowest mean CV MAE (1.025 vs 1.047 D
   for z-scores), is best at every N for both scale-sensitive models, and wins 45% of the 108 paired cells. The gain
   is small (≈ 0.02 D) but consistent.
4. **Target chosen: √|μ|** (mean CV MAE 0.797 vs 0.809 D). It helps ridge and kernel ridge at every N (0.01–0.03 D)
   and is mixed for XGBoost (worse at N = 100, better at 300 and 1000). RMSE is also slightly lower.
5. **PCA keeps variance, not relevance.** 8 components keep 88% of the Coulomb spectrum's variance but only 74% of
   cm+structure's, and the two components that track |μ| best (PC 9, PC 10) fall just outside the first 8.
   Compression is nearly free on the spectrum (+0.003 to +0.008 D) but costs +0.13 D (kernel ridge) to +0.31 D
   (ridge) on cm+structure at N = 1000.
6. **Effective dimension.** The 29 Coulomb eigenvalues behave like 4.7 independent directions (participation
   ratio), the 31 structure features like 9.3, and cm+structure like 8.5 (27 components for 95% of the variance).
   Collinearity is strong (16 structure features with VIF > 10; n_C 76), so ridge rather than OLS. After
   Yeo-Johnson no feature is an exact linear combination of the others: the per-column power transform breaks
   identities such as n_H = Σ H-bond counts.
7. **For the quantum model (M4).** The 8 component scores have unequal spreads (sd 3.3–4.0 for PC 1 vs 0.7–1.3 for
   PC 8), so each should be re-standardized before the angle map γ·x, or PC 1 will dominate. Avoid the quantile
   map. Feeding richer features through 8 principal components loses signal, so which 8 numbers to encode is an
   M4 design question.

**Carried forward:** notebooks 04 and 05 use Yeo-Johnson scaling and the √|μ| target for every model
(`results/explore03_preprocessing.json`).